# 02 Team Game Predictor

This notebook builds one row per game, creates rolling pre-game features, trains baseline score and winner models, and saves artifacts into `models/`.

Important distinction: this notebook is team-level. The model predicts whether the home team wins and estimates home/away scores from rolling team performance features. Player-level data is handled separately in `03_player_event_weights.ipynb` and is used by the simulator to assign events to players.


## Load And Identify Games

This cell loads the team warehouse and creates the identifiers needed to pair the home and away rows for each game. It also derives `IS_HOME` from the matchup string.


In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_absolute_error

# Resolve paths so the notebook works from either the repo root or notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
WAREHOUSE = ROOT / 'warehouse'
MODELS = ROOT / 'models'
MODELS.mkdir(exist_ok=True)

# Load the team-game warehouse. This table has one row per team per game.
team = pd.read_csv(WAREHOUSE / 'fact_team_game.csv')
team['GAME_DATE'] = pd.to_datetime(team['GAME_DATE'])
# GAME_KEY groups the two team rows that belong to the same NBA game.
team['GAME_KEY'] = team['SEASON'].astype(str) + '_' + team['SEASON_TYPE'].astype(str) + '_' + team['GAME_ID'].astype(str)
# NBA matchup strings use 'vs.' for home games and '@' for away games.
team['IS_HOME'] = team['MATCHUP'].str.contains(' vs. ', regex=False)
team = team.sort_values(['GAME_DATE', 'GAME_ID']).drop_duplicates(['GAME_KEY', 'TEAM_ABBREVIATION'])
team.head()


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,STL,BLK,TOV,PF,PLUS_MINUS,SEASON,SEASON_TYPE,TEAM_ABBR,GAME_KEY,IS_HOME
0,22015,1610612737,ATL,Atlanta Hawks,21500001,2015-10-27,ATL vs. DET,L,239,94,...,9,4,15,25,-12.0,2015-16,Regular Season,ATL,2015-16_Regular Season_21500001,True
4,22015,1610612765,DET,Detroit Pistons,21500001,2015-10-27,DET @ ATL,W,239,106,...,5,3,15,15,12.0,2015-16,Regular Season,DET,2015-16_Regular Season_21500001,False
1,22015,1610612741,CHI,Chicago Bulls,21500002,2015-10-27,CHI vs. CLE,W,240,97,...,6,10,13,22,2.0,2015-16,Regular Season,CHI,2015-16_Regular Season_21500002,True
2,22015,1610612739,CLE,Cleveland Cavaliers,21500002,2015-10-27,CLE @ CHI,L,240,95,...,5,7,10,21,-2.0,2015-16,Regular Season,CLE,2015-16_Regular Season_21500002,False
3,22015,1610612740,NOP,New Orleans Pelicans,21500003,2015-10-27,NOP @ GSW,L,241,95,...,9,3,18,26,-16.0,2015-16,Regular Season,NOP,2015-16_Regular Season_21500003,False


## Build Rolling Team Features

For each team, this section creates pre-game rolling features. The `shift(1)` is important: it prevents leakage by making sure a game row only uses statistics from earlier games.


In [2]:
# These are team-level box-score stats used to describe form entering a game.
stat_cols = ['PTS', 'PLUS_MINUS', 'FG_PCT', 'FG3_PCT', 'FT_PCT', 'FGA', 'FG3A', 'FTA', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF']

rolling = team[['GAME_KEY', 'GAME_DATE', 'TEAM_ABBREVIATION'] + stat_cols].copy()
rolling = rolling.sort_values(['TEAM_ABBREVIATION', 'GAME_DATE', 'GAME_KEY'])

for col in stat_cols:
    rolling[f'{col}_last5'] = (
        rolling.groupby('TEAM_ABBREVIATION')[col]
        # Shift by one game so a row never uses its own final score/stat line as input.
        .transform(lambda s: s.shift(1).rolling(5, min_periods=3).mean())
    )
    rolling[f'{col}_season_avg'] = (
        rolling.groupby('TEAM_ABBREVIATION')[col]
        # Expanding season average gives a longer-term baseline, also only using prior games.
        .transform(lambda s: s.shift(1).expanding(min_periods=5).mean())
    )

feature_cols = [c for c in rolling.columns if c.endswith('_last5') or c.endswith('_season_avg')]
rolling_features = rolling[['GAME_KEY', 'TEAM_ABBREVIATION'] + feature_cols]
rolling_features.head()


,GAME_KEY,TEAM_ABBREVIATION,PTS_last5,PTS_season_avg,PLUS_MINUS_last5,PLUS_MINUS_season_avg,FG_PCT_last5,FG_PCT_season_avg,FG3_PCT_last5,FG3_PCT_season_avg,...,AST_last5,AST_season_avg,STL_last5,STL_season_avg,BLK_last5,BLK_season_avg,TOV_last5,TOV_season_avg,PF_last5,PF_season_avg
0,2015-16_Regular Season_21500001,ATL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
39,2015-16_Regular Season_21500019,ATL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60,2015-16_Regular Season_21500026,ATL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89,2015-16_Regular Season_21500039,ATL,101.00,NaN,0.666667,NaN,0.463667,NaN,0.353667,NaN,...,23.666667,NaN,9.666667,NaN,4.0,NaN,15.0,NaN,20.0,NaN
115,2015-16_Regular Season_21500055,ATL,99.25,NaN,1.000000,NaN,0.452750,NaN,0.325500,NaN,...,23.250000,NaN,9.500000,NaN,4.5,NaN,14.0,NaN,19.0,NaN


## Create One Row Per Game

The warehouse has one row per team-game. The model needs one row per actual game, so home rows and away rows are merged together, then home/away rolling features are attached.


In [3]:
# Split the two team rows for each game into home and away sides.
home = team[team['IS_HOME']].copy()
away = team[~team['IS_HOME']].copy()

games = home.merge(away, on='GAME_KEY', suffixes=('_HOME', '_AWAY'))
games = games[['GAME_KEY', 'GAME_DATE_HOME', 'SEASON_HOME', 'SEASON_TYPE_HOME', 'TEAM_ABBREVIATION_HOME', 'TEAM_ABBREVIATION_AWAY', 'PTS_HOME', 'PTS_AWAY', 'WL_HOME']]
games = games.rename(columns={
    'GAME_DATE_HOME': 'GAME_DATE',
    'SEASON_HOME': 'SEASON',
    'SEASON_TYPE_HOME': 'SEASON_TYPE',
    'TEAM_ABBREVIATION_HOME': 'HOME_TEAM',
    'TEAM_ABBREVIATION_AWAY': 'AWAY_TEAM',
    'PTS_HOME': 'HOME_SCORE',
    'PTS_AWAY': 'AWAY_SCORE',
})
# Target for the classifier: 1 if the home team won, 0 if the away team won.
games['HOME_WIN'] = games['WL_HOME'].eq('W').astype(int)
games['MARGIN'] = games['HOME_SCORE'] - games['AWAY_SCORE']

# Prefix the same rolling features for each side so the model sees both teams.
home_features = rolling_features.add_prefix('HOME_').rename(columns={'HOME_GAME_KEY': 'GAME_KEY', 'HOME_TEAM_ABBREVIATION': 'HOME_TEAM'})
away_features = rolling_features.add_prefix('AWAY_').rename(columns={'AWAY_GAME_KEY': 'GAME_KEY', 'AWAY_TEAM_ABBREVIATION': 'AWAY_TEAM'})

# Final modeling table: one row per game with home features, away features, and labels.
dataset = games.merge(home_features, on=['GAME_KEY', 'HOME_TEAM']).merge(away_features, on=['GAME_KEY', 'AWAY_TEAM'])
dataset = dataset.dropna().sort_values('GAME_DATE')
dataset.shape


(13951, 67)

## Train And Evaluate Team-Level Models

The split is chronological: older games train the models, newer games test them. The classifier predicts `HOME_WIN`; the regressor estimates the home and away scores.


In [4]:
# Use only rolling pre-game features as model inputs. Scores and labels stay out.
model_features = [c for c in dataset.columns if c.startswith('HOME_') and (c.endswith('_last5') or c.endswith('_season_avg'))]
model_features += [c for c in dataset.columns if c.startswith('AWAY_') and (c.endswith('_last5') or c.endswith('_season_avg'))]

# Time-based split: train on the oldest 80% of games and test on newer games.
# This is more realistic than a random split because future games should not help
# predict past games.
split_date = dataset['GAME_DATE'].quantile(0.80)
train = dataset[dataset['GAME_DATE'] <= split_date]
test = dataset[dataset['GAME_DATE'] > split_date]

X_train = train[model_features]
X_test = test[model_features]

# Random Forests are strong baselines for tabular sports data and require little
# feature scaling. The classifier predicts HOME_WIN; the regressor predicts scores.
winner_model = RandomForestClassifier(n_estimators=300, random_state=42, min_samples_leaf=10, n_jobs=-1)
score_model = RandomForestRegressor(n_estimators=300, random_state=42, min_samples_leaf=10, n_jobs=-1)

winner_model.fit(X_train, train['HOME_WIN'])
score_model.fit(X_train, train[['HOME_SCORE', 'AWAY_SCORE']])

winner_pred = winner_model.predict(X_test)
score_pred = score_model.predict(X_test)

# Accuracy measures how often the classifier picks the correct winner on held-out games.
print('Winner accuracy:', round(accuracy_score(test['HOME_WIN'], winner_pred), 3))
print('Home score MAE:', round(mean_absolute_error(test['HOME_SCORE'], score_pred[:, 0]), 2))
print('Away score MAE:', round(mean_absolute_error(test['AWAY_SCORE'], score_pred[:, 1]), 2))
print('Margin MAE:', round(mean_absolute_error(test['MARGIN'], score_pred[:, 0] - score_pred[:, 1]), 2))


Winner accuracy: 0.623
Home score MAE: 9.97
Away score MAE: 9.9
Margin MAE: 12.24


## Save Model Artifacts

The artifact stores both models, the feature list, and the split date. The saved CSV preserves the modeling dataset so the reported metrics can be recreated later.


In [5]:
# Save the trained models, feature list, and split date so the exact experiment
# can be inspected or reused by app code later.
artifacts = {
    'winner_model': winner_model,
    'score_model': score_model,
    'features': model_features,
    'split_date': split_date,
}
joblib.dump(artifacts, MODELS / 'team_game_predictor.joblib')
dataset.to_csv(MODELS / 'team_game_training_dataset.csv', index=False)
MODELS / 'team_game_predictor.joblib'


WindowsPath('c:/Users/jorda/Programs/nba_ml/models/team_game_predictor.joblib')